# Build 5 - Synthetic Generator Investigation

**Objective:** characterize whether the Playground S6E8 synthetic dataset
contains repeatable generator structure that is real, measurable,
understandable, relevant to modeling/validation, rules-compliant, and
strong enough to justify controlled experiments. This is a forensic
investigation, not a feature-tuning pass -- descriptive/exact-value
diagnostics first, formal modeling only if Phase A finds a strong,
explainable, rules-compliant candidate.

Out of scope: hyperparameter tuning (Build 6), ensembling/blending
(Build 7), final submission strategy (Build 9).

## 1. Scope and frozen controls

**Frozen control entering Build 5** (Build 4, `docs/DECISIONS.md`,
`experiments/experiments.csv`):

| | |
|---|---|
| Experiment | E006 - XGBoost + `screen_residual` |
| CV mean | 0.96445 |
| CV std | 0.00056 |
| Public LB | 0.96608 |
| Feature set | raw predictors + `screen_residual` |

Build 5 does not change this control unless a Build 5 experiment is
separately validated against it. No hyperparameter tuning, no iteration-
budget expansion, no ensembling in this notebook.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

import sys
sys.path.insert(0, "..")
from src.config import (
    TRAIN_PATH, TEST_PATH, OUTPUTS_DIR, NUMERIC_COLS, CATEGORICAL_COLS,
    TARGET_COLUMN, ID_COLUMN,
)
from src.features import SCREEN_TIME_COMPONENT_COLS

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 25)

PREDICTOR_COLS = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(train.shape, test.shape)
GLOBAL_RATE = train[TARGET_COLUMN].mean()
print("global train target rate:", round(GLOBAL_RATE, 4))

(691369, 14) (296302, 13)
global train target rate: 0.7094


## 2. Revisit Build 1 synthetic findings (Phase A1)

Summarizing only what prior evidence already established (not re-deriving
from scratch), drawn from `notebooks/01_eda.ipynb` and
`docs/COMPETITION_NOTES.md`:

- **Screen-time composition:** `daily_screen_time_hours >= social_media_hours
  + gaming_hours + work_study_hours` holds in 100% of the 421,427
  fully-observed train rows (Section 11). The residual is right-skewed,
  non-negative, and larger for `addicted_label == 1`.
- **`screen_residual`:** accepted into the Build 4+ feature set (E006,
  +0.00062 CV vs the XGBoost control, 5/5 folds, transfers to CatBoost at a
  near-identical magnitude in E008).
- **Missingness:** every predictor has 4.2%-19.4% missingness in train;
  small train/test gaps (largest 3.4pp); not meaningfully related to the
  target per-column or via total missing count; missingness-indicator
  correlations across columns are all small (Sections 4.2, 12).
- **Duplicates:** no predictor-vector duplicates within train or within
  test; only 2 negligible train/test overlaps, both explained by
  near-total-missingness rows (Section 5).
- **`id`:** sequential, contiguous, no target relationship or drift across
  10-20 `id` bins (Section 13). Excluded from the feature set.
- **Train/test shift:** no significant KS-test difference on any numeric
  feature, no meaningful categorical proportion shift (Section 9).

Build 5 treats none of this as proven mechanism -- only as prior evidence
to build on, verify, and extend.

## 3. Value precision and quantization (Phase A2)

For each numeric predictor: unique-value count, dominant precision grid,
minimum gap between sorted unique values, and boundary mass (fraction of
rows sitting exactly at the observed min or max).

In [2]:
def quantization_audit(train_df, test_df, cols):
    rows = []
    for col in cols:
        tr = train_df[col].dropna()
        te = test_df[col].dropna()
        tr_vals = np.sort(tr.unique())
        te_vals = np.sort(te.unique())
        diffs = np.diff(tr_vals)

        def frac_match(vals, step):
            return float(np.mean(np.isclose(vals, np.round(vals / step) * step, atol=1e-6)))

        fracs = {step: frac_match(tr_vals, step) for step in (1, 0.1, 0.01)}
        # threshold over *unique* values, not row-weighted: a handful of
        # off-grid unique values (float noise) can register well below 1.0
        # even when they cover a negligible fraction of rows.
        dominant = next((step for step in (1, 0.1, 0.01) if fracs[step] > 0.995), None)
        boundary_mass = float(((tr == tr.min()) | (tr == tr.max())).mean())

        rows.append({
            "feature": col,
            "n_unique_train": len(tr_vals),
            "n_unique_test": len(te_vals),
            "dominant_precision": dominant,
            "min_gap": round(float(diffs.min()), 5) if len(diffs) else None,
            "frac_on_dominant_grid": round(fracs[dominant], 6) if dominant else None,
            "boundary_mass": round(boundary_mass, 5),
            "min_train": float(tr.min()), "max_train": float(tr.max()),
            "min_test": float(te.min()), "max_test": float(te.max()),
        })
    return pd.DataFrame(rows)


quant = quantization_audit(train, test, NUMERIC_COLS)
quant

,feature,n_unique_train,n_unique_test,dominant_precision,min_gap,frac_on_dominant_grid,boundary_mass,min_train,max_train,min_test,max_test
0,age,18,18,1.00,1.0000,1.000000,0.10234,18.00,35.00,18.00,35.00
1,daily_screen_time_hours,1389,1349,0.01,0.0020,0.998560,0.00044,0.50,15.00,0.50,15.00
2,social_media_hours,721,703,0.01,0.0100,1.000000,0.00002,0.00,8.00,0.00,7.85
3,gaming_hours,401,401,0.01,0.0100,1.000000,0.00021,0.00,4.00,0.00,4.00
4,work_study_hours,600,601,0.01,0.0100,1.000000,0.00002,0.00,6.00,0.00,6.00
5,sleep_hours,451,451,0.01,0.0100,1.000000,0.00029,4.50,9.00,4.50,9.00
6,notifications_per_day,231,231,1.00,1.0000,1.000000,0.00629,20.00,250.00,20.00,250.00
7,app_opens_per_day,166,166,1.00,1.0000,1.000000,0.01097,15.00,180.00,15.00,180.00
8,weekend_screen_time,1437,1404,0.01,0.0035,0.999304,0.00003,0.51,17.56,0.51,17.56


**Observation:**

- `age`, `notifications_per_day`, `app_opens_per_day` sit on an exact
  integer grid (dominant precision 1, 100% match). The latter two are only
  stored as `float64` because the column also contains `NaN`.
- All six continuous hour-based features sit on a 0.01 grid at >99.9%
  match. `daily_screen_time_hours` and `weekend_screen_time` have a
  handful of off-grid *unique* values (2 and 3 rows respectively, out of
  595,515 and 579,306 non-null rows) -- negligible float-representation
  noise, not a second precision regime.
- Train and test ranges match exactly for every column (already known from
  Build 1); boundary mass is small everywhere (<10.3%, dominated by `age`'s
  uniform integer range), no evidence of clipping-driven spikes at the
  observed min/max beyond what a bounded generator (e.g. `sleep_hours` in
  [4.5, 9.0]) would produce on its own.

Artifact: `outputs/numeric_quantization_audit.csv`.

In [3]:
quant.to_csv(OUTPUTS_DIR / "numeric_quantization_audit.csv", index=False)

## 4. Screen-time arithmetic structure (Phase A3)

Build 4 confirmed `screen_residual` adds signal. Characterizing *why*: the
residual's distribution, precision, train/test consistency, and
relationship to the target and to other features.

In [4]:
def with_residual(df):
    sub = df.dropna(subset=["daily_screen_time_hours"] + SCREEN_TIME_COMPONENT_COLS).copy()
    sub["component_sum"] = sub[SCREEN_TIME_COMPONENT_COLS].sum(axis=1)
    sub["screen_residual"] = sub["daily_screen_time_hours"] - sub["component_sum"]
    return sub


tr_resid = with_residual(train)
te_resid = with_residual(test)
print("train complete rows:", len(tr_resid), f"({len(tr_resid) / len(train):.1%})")
print("test complete rows:", len(te_resid), f"({len(te_resid) / len(test):.1%})")

r = tr_resid["screen_residual"].round(6)
print("\nn unique residual values (train):", r.nunique())
print("prop residual < 0 (tol 1e-3):", (r < -1e-3).mean())
print("prop residual == 0 (tol 1e-6):", round((r.abs() < 1e-6).mean(), 6))
print("prop residual > 0:", round((r > 1e-6).mean(), 6))
r.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])

train complete rows: 421427 (61.0%)
test complete rows: 182287 (61.5%)

n unique residual values (train): 1022
prop residual < 0 (tol 1e-3): 0.0
prop residual == 0 (tol 1e-6): 0.000942
prop residual > 0: 0.999058


count    421427.000000
mean          1.342654
std           1.480499
min           0.000000
1%            0.080000
5%            0.220000
25%           0.460000
50%           0.750000
75%           1.460000
95%           4.760000
99%           6.830000
max          11.530000
Name: screen_residual, dtype: float64

In [5]:
# residual precision: is it on the same 0.01 grid as the raw features?
diffs = np.diff(np.sort(r.unique()))
frac_on_001_grid = np.mean(np.isclose(r.unique(), np.round(r.unique() / 0.01) * 0.01, atol=1e-6))
print("min gap between sorted unique residuals:", round(diffs.min(), 5))
print("fraction of unique residual values on the 0.01 grid:", round(frac_on_001_grid, 6))

min gap between sorted unique residuals: 0.01
fraction of unique residual values on the 0.01 grid: 1.0


In [6]:
print("residual by target:")
print(tr_resid.groupby(TARGET_COLUMN)["screen_residual"].agg(["mean", "median", "std", "count"]))

residual by target:
                    mean  median       std   count
addicted_label                                    
0               0.637453    0.48  0.627110  122540
1               1.631777    0.95  1.625363  298887


In [7]:
te_r = te_resid["screen_residual"].round(6)
ks_result = stats.ks_2samp(r, te_r)
print("train residual: mean", round(r.mean(), 5), "median", round(r.median(), 5))
print("test residual:  mean", round(te_r.mean(), 5), "median", round(te_r.median(), 5))
print("KS train vs test residual:", ks_result)

train residual: mean 1.34265 median 0.75
test residual:  mean 1.33923 median 0.75
KS train vs test residual: KstestResult(statistic=np.float64(0.0025266421433106645), pvalue=np.float64(0.3903010292400876), statistic_location=np.float64(0.78), statistic_sign=np.int8(-1))


In [8]:
print("residual by gender:")
print(tr_resid.groupby("gender")["screen_residual"].agg(["mean", "median", "count"]))
print()
print("residual by stress_level:")
print(tr_resid.groupby("stress_level")["screen_residual"].agg(["mean", "median", "count"]))

residual by gender:
            mean  median   count
gender                          
Female  1.329979    0.75  136092
Male    1.369672    0.77  136915
Other   1.326854    0.75  132933

residual by stress_level:


                  mean  median   count
stress_level                          
High          1.330782    0.76  136241
Low           1.347319    0.75  128231
Medium        1.348115    0.74  128221


In [9]:
dsth_bins = pd.qcut(tr_resid["daily_screen_time_hours"], 10, duplicates="drop")
tr_resid.groupby(dsth_bins, observed=True)["screen_residual"].agg(["mean", "median", "count"])

,mean,median,count
daily_screen_time_hours,,,
"(0.499, 3.97]",0.310545,0.29,42311
"(3.97, 4.97]",0.471642,0.45,42076
"(4.97, 5.9]",0.605434,0.56,42694
"(5.9, 6.8]",0.753207,0.66,41560
"(6.8, 7.77]",0.930304,0.77,42221
"(7.77, 8.69]",1.165731,0.91,42045
"(8.69, 9.47]",1.559226,1.11,43145
"(9.47, 10.25]",1.913049,1.33,41778
"(10.25, 11.24]",2.409396,1.75,42609


**Observation:**

1. **Does the residual occupy a small discrete set?** No. 1,022 distinct
   values across 421,427 complete rows, spanning 0.00-11.53h.
2. **Is it nearly deterministic?** No -- continuous-looking, right-skewed
   (mean 1.34h, median 0.75h), essentially never negative and only exactly
   zero for 0.09% of rows.
3. **Grid:** the residual sits on the same 0.01 grid as the raw hour
   features (100% of unique values), consistent with it being a *derived*
   quantity, not independently rounded.
4. **Target relationship:** clearly separated group means (0.637h for
   `addicted_label == 0` vs 1.632h for `addicted_label == 1`) -- this is
   the mechanism behind `screen_residual`'s accepted Build 4 gain.
5. **Train/test consistency:** KS test p ≈ 0.39 (not significant) -- no
   detectable shift, consistent with Build 1's broader train/test shift
   finding.
6. **Gender/stress_level:** both flat (means within ~0.02h of each other
   across categories) -- the residual is not concentrated in a
   demographic subgroup.
7. **Correlated with `daily_screen_time_hours` itself** (mean rises from
   0.31h in the lowest decile to 3.36h in the highest) -- mechanically
   expected, since the residual is defined as part of that total.

**Implication:** the residual behaves like a genuine, continuously-drawn
"other/unaccounted" screen-time component added on top of the three named
components at generation time, not a discrete template or lookup value.
This explains, rather than extends, why `screen_residual` helps E006: it
recovers real information the three named components alone do not carry.

## 5. Repeated-value and frequency analysis (Phase A4 + A5)

**Phase A4** -- exact-value target-rate analysis with a minimum-support
threshold (200 rows) to avoid small-sample artifacts. **Phase A5** --
value-frequency stability between train and test, and its relationship
(if any) to the target.

In [10]:
MIN_SUPPORT = 200


def exact_value_target_rate(df, col, min_support=MIN_SUPPORT):
    g = df.groupby(col, observed=True)[TARGET_COLUMN].agg(["mean", "count"])
    g = g[g["count"] >= min_support].sort_values("mean")
    g["delta_vs_global"] = g["mean"] - GLOBAL_RATE
    return g


residual_rates = exact_value_target_rate(tr_resid, "screen_residual")
print("screen_residual exact values with support >= 200:", len(residual_rates))
print("target-rate spread among them:", round(residual_rates["mean"].max() - residual_rates["mean"].min(), 4))
spearman_corr = residual_rates.reset_index()["screen_residual"].corr(
    residual_rates.reset_index()["mean"], method="spearman"
)
print("Spearman corr(exact residual value, target rate):", round(spearman_corr, 4))
residual_rates.head(5)

screen_residual exact values with support >= 200: 405
target-rate spread among them: 0.7843
Spearman corr(exact residual value, target rate): 0.7506


,mean,count,delta_vs_global
screen_residual,,,
0.15,0.211066,488,-0.498359
0.14,0.211302,407,-0.498122
0.19,0.231250,320,-0.478174
0.29,0.239544,263,-0.469881
0.26,0.240591,744,-0.468833


In [11]:
for col in ["stress_level", "academic_work_impact", "gender"]:
    print(f"--- {col} ---")
    print(exact_value_target_rate(train, col))
    print()

--- stress_level ---


                  mean   count  delta_vs_global
stress_level                                   
Medium        0.705663  207565        -0.003761
Low           0.711160  207783         0.001736
High          0.711431  220873         0.002007

--- academic_work_impact ---


                          mean   count  delta_vs_global
academic_work_impact                                   
Yes                   0.707883  330566        -0.001542
No                    0.711029  316579         0.001605

--- gender ---
            mean   count  delta_vs_global
gender                                   
Other   0.701020  217078        -0.008404
Female  0.703838  221595        -0.005586
Male    0.723198  223662         0.013774



In [12]:
train["dsth_round_0_5"] = (train["daily_screen_time_hours"] / 0.5).round() * 0.5
dsth_rates = exact_value_target_rate(train, "dsth_round_0_5")
dsth_rates

,mean,count,delta_vs_global
dsth_round_0_5,,,
1.0,0.209224,889,-0.500200
2.5,0.219629,8528,-0.489795
0.5,0.219780,364,-0.489644
2.0,0.228724,5981,-0.480700
1.5,0.232789,2513,-0.476635
3.0,0.242640,12772,-0.466784
3.5,0.284395,18302,-0.425029
4.0,0.285549,28573,-0.423875
4.5,0.330459,30503,-0.378965


**Observation (Phase A4):** the `screen_residual` exact-value target rate
rises smoothly and monotonically with the residual value (Spearman
0.75) -- a real, strong relationship, but a smooth gradient rather than an
abrupt staircase between adjacent values, and consistent throughout with
the residual being a continuous quantity (Section 4), not a set of
discrete generator "codes". `stress_level`, `academic_work_impact`, and
`gender` all show weak deltas (<1.4 percentage points from the global
rate). Rounded `daily_screen_time_hours` (0.5h bins) shows the dataset's
strongest and smoothest relationship of all: target rate rises from ~21%
at 1.0h to 100% at 13-14h, all bins with support >= 364. This is the
dataset's dominant signal (already reflected in every model's feature
importances since Build 3) and is treated here as an **ordinary
predictive relationship**, not a hidden artifact -- gradient-boosted trees
already model it optimally as a sequence of continuous splits.

In [13]:
for col in ["age", "notifications_per_day", "app_opens_per_day"]:
    vc_tr = train[col].value_counts()
    vc_te = test[col].value_counts()
    rate = train.groupby(col, observed=True)[TARGET_COLUMN].mean()
    joined = pd.DataFrame({"freq": vc_tr, "rate": rate}).dropna()
    freq_corr = joined["freq"].corr(joined["rate"], method="spearman")
    tr_norm, te_norm = vc_tr / vc_tr.sum(), vc_te / vc_te.sum()
    stability = pd.DataFrame({"train": tr_norm, "test": te_norm}).dropna()
    train_test_corr = stability["train"].corr(stability["test"])
    print(
        f"{col}: n_values={len(vc_tr)}, "
        f"corr(freq, target rate)={freq_corr:.3f}, "
        f"corr(train freq, test freq)={train_test_corr:.4f}"
    )

age: n_values=18, corr(freq, target rate)=-0.240, corr(train freq, test freq)=0.9948
notifications_per_day: n_values=231, corr(freq, target rate)=0.009, corr(train freq, test freq)=0.9987


app_opens_per_day: n_values=166, corr(freq, target rate)=0.091, corr(train freq, test freq)=0.9983


**Observation (Phase A5):** value-frequency distributions are highly
stable between train and test for all three checked columns (correlation
0.995-0.999) -- expected from a stationary generator, not evidence of an
exploitable train-vs-test frequency gap. Frequency itself carries no
independent target signal beyond what the value already carries
(`notifications_per_day`/`app_opens_per_day` correlations near zero;
`age`'s -0.24 is fully explained by age's own weak target relationship,
not by frequency). No frequency-encoding candidate identified.

## 6. Cross-feature repeated patterns (Phase A6)

Hashing/grouping on two semantically-justified subsets: the four
screen-time columns, and the three low-cardinality categoricals.

In [14]:
cols6 = ["social_media_hours", "gaming_hours", "work_study_hours", "daily_screen_time_hours"]
sub6 = train.dropna(subset=cols6)
combo_hash = pd.util.hash_pandas_object(sub6[cols6], index=False)
combo_counts = combo_hash.value_counts()
print("unique combinations:", len(combo_counts), "of", len(sub6), "rows")
combo_counts.describe()

unique combinations: 421384 of 421427 rows


count    421384.000000
mean          1.000102
std           0.010333
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           3.000000
Name: count, dtype: float64

In [15]:
categorical_combo = ["stress_level", "academic_work_impact", "gender"]
combo_sizes = train.groupby(categorical_combo, observed=True).size().sort_values(ascending=False)
combo_rate = train.groupby(categorical_combo, observed=True)[TARGET_COLUMN].agg(["mean", "count"])
combo_rate.sort_values("count", ascending=False)

mean  count
stress_level academic_work_impact gender                 
High         Yes                  Other   0.699246  36332
                                  Female  0.703142  34562
                                  Male    0.724616  34490
Medium       Yes                  Male    0.717627  32535
Low          Yes                  Male    0.725710  32411
                                  Female  0.706020  32390
             No                   Male    0.723998  32344
High         No                   Other   0.707120  32218
Medium       No                   Male    0.720265  32041
Low          No                   Female  0.707594  32024
High         No                   Male    0.727370  31849
Medium       Yes                  Female  0.698778  31671
High         No                   Female  0.708530  31499
Medium       No                   Female  0.698451  31431
             Yes                  Other   0.699355  30847
Low          Yes                  Other   0.698677  30532
Medium       No                   Other   0.700741  30238
Low          No                   Other   0.701105  29589

**Observation:** the 4-column screen-time combination is 99.99% unique
(421,384 of 421,427 rows; largest repeat group is 3 rows) -- no template
reuse at native precision. The 3-way categorical combination (18 cells) has
near-uniform cell sizes (~29.6k-36.3k, as expected from three roughly
independent categoricals) and target rates confined to a narrow 0.698-0.727
band with no interaction structure. No compact "template" pattern found in
either subset.

## 7. Near-duplicate investigation (Phase A7)

Section 3/6 established the screen-time features sit on a 0.01 grid with
almost no native-precision repeats. Testing a coarser, justified rounding
(0.1h, one order of magnitude coarser than the observed grid) to check
whether *near*-duplicates carry consistent target labels.

In [16]:
rounded = sub6[cols6].round(1)
group_key = rounded.apply(tuple, axis=1)
grouped = sub6.assign(_key=group_key.values).groupby("_key")
group_sizes = grouped.size()
print("n groups (0.1h rounding):", len(group_sizes), "of", len(sub6), "rows")
print("group size distribution:")
print(group_sizes.describe())

big_groups = group_sizes[group_sizes >= 5].index
group_target_rate = grouped[TARGET_COLUMN].mean().loc[big_groups]
print(
    "\ntarget-rate spread within groups of >=5 rows: "
    f"min={group_target_rate.min():.3f}, max={group_target_rate.max():.3f}, "
    f"mean={group_target_rate.mean():.3f}"
)

n groups (0.1h rounding): 321821 of 421427 rows
group size distribution:
count    321821.000000
mean          1.309507
std           0.877655
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
dtype: float64



target-rate spread within groups of >=5 rows: min=0.000, max=1.000, mean=0.380


**Observation:** 321,821 groups from 421,427 rows (mean group size 1.31,
max 21) -- rounding to 0.1h produces mostly coincidental singleton/small
clusters, not meaningful repeated profiles. Target rate within groups of
5+ rows spans the *entire* possible range (0.0 to 1.0) -- as wide as the
whole population. This argues against, not for, a reusable near-duplicate
structure: a target-propagation feature built on this rounding would fit
group-level sampling noise, not signal.

## 8. Missingness generator structure (Phase A8)

Build 1 found per-column missingness uninformative and pairwise
missingness-indicator correlations small. Extending this to the full joint
missingness pattern (a 12-column presence/absence mask per row).

In [17]:
missing_mask = train[PREDICTOR_COLS].isna()
pattern = missing_mask.apply(tuple, axis=1)
pattern_counts = pattern.value_counts()
print("n unique missingness patterns (train):", len(pattern_counts))

missing_mask_test = test[PREDICTOR_COLS].isna()
pattern_test = missing_mask_test.apply(tuple, axis=1)
pattern_counts_test = pattern_test.value_counts()
print("n unique missingness patterns (test):", len(pattern_counts_test))

overlap = len(set(pattern_counts.head(15).index) & set(pattern_counts_test.head(15).index))
print("overlap of top-15 patterns between train and test:", overlap, "/ 15")

n unique missingness patterns (train): 2925


n unique missingness patterns (test): 2522
overlap of top-15 patterns between train and test: 14 / 15


In [18]:
rows = []
for pat, cnt in pattern_counts.head(15).items():
    missing_cols = [c for c, m in zip(PREDICTOR_COLS, pat) if m]
    rate = train.loc[pattern == pat, TARGET_COLUMN].mean()
    rows.append({
        "missing_columns": ", ".join(missing_cols) or "(none)",
        "count": cnt,
        "share_of_train": round(cnt / len(train), 4),
        "target_rate": round(rate, 4),
    })
pd.DataFrame(rows)

,missing_columns,count,share_of_train,target_rate
0,(none),269185,0.3894,0.7081
1,social_media_hours,22073,0.0319,0.7057
2,app_opens_per_day,22056,0.0319,0.7132
3,gaming_hours,20006,0.0289,0.7100
4,work_study_hours,17185,0.0249,0.7080
5,notifications_per_day,17119,0.0248,0.7061
6,weekend_screen_time,16428,0.0238,0.7117
7,stress_level,13874,0.0201,0.7098
8,daily_screen_time_hours,13183,0.0191,0.7068
9,"social_media_hours, gaming_hours",12658,0.0183,0.7070


In [19]:
row_missing_count = missing_mask.sum(axis=1)
train.groupby(row_missing_count)[TARGET_COLUMN].agg(["mean", "count"])

,mean,count
0,0.708123,269185
1,0.708970,180459
2,0.711178,120697
3,0.711784,67328
4,0.709494,32557
5,0.712458,13678
6,0.719217,5157
7,0.696301,1676
8,0.702306,477
9,0.661765,136


**Observation:**

- 2,925 unique missingness patterns realized in train (2,522 in test) out
  of a combinatorial ceiling near 2^12 = 4,096 -- consistent with
  near-independent per-column missingness (matches Build 1's pairwise
  correlation finding), not a small library of reused missingness
  templates.
- Target rate across the top 15 patterns (covering the large majority of
  rows) stays in a flat 0.696-0.716 band.
- `row_missing_count` target rate stays in 0.696-0.719 for counts 0-8
  (support 477-269,185 rows); only counts >=9 drift further, but on tiny
  samples (<=136 rows) not distinguishable from noise.
- 14 of the top 15 train patterns also appear in test's top 15 -- stable
  pattern distribution across splits.

**Implication:** no evidence that missingness pattern (individually or
jointly) is target-informative or template-driven. This reconfirms and
extends Build 1's decision that missingness is not treated as informative
by default -- no missingness-pattern feature is justified.

## 9. ID/order/batch structure (Phase A9)

Build 1 checked raw feature means, missingness rate, and target rate
across `id` bins and found no drift. Extending the check to the two
Build 4/5-relevant *derived* quantities: `screen_residual` and
`row_missing_count`.

In [20]:
id_bins20 = pd.qcut(tr_resid[ID_COLUMN], 20, duplicates="drop")
tr_resid.groupby(id_bins20, observed=True)["screen_residual"].agg(["mean", "count"])

,mean,count
id,,
"(2.999, 34474.6]",1.349212,21072
"(34474.6, 69155.6]",1.347674,21071
"(69155.6, 103738.9]",1.316816,21071
"(103738.9, 138265.4]",1.341738,21072
"(138265.4, 172724.5]",1.345268,21071
"(172724.5, 207142.6]",1.348855,21071
"(207142.6, 241680.1]",1.347706,21072
"(241680.1, 276350.8]",1.349606,21071
"(276350.8, 311077.7]",1.350756,21071


In [21]:
id_bins20_full = pd.qcut(train[ID_COLUMN], 20, duplicates="drop")
train.assign(_row_missing=row_missing_count).groupby(id_bins20_full, observed=True)["_row_missing"].mean()

id
(-0.001, 34568.4]       1.260002
(34568.4, 69136.8]      1.261919
(69136.8, 103705.2]     1.263820
(103705.2, 138273.6]    1.254773
(138273.6, 172842.0]    1.262345
(172842.0, 207410.4]    1.248409
(207410.4, 241978.8]    1.257059
(241978.8, 276547.2]    1.255807
(276547.2, 311115.6]    1.263567
(311115.6, 345684.0]    1.257398
(345684.0, 380252.4]    1.254802
(380252.4, 414820.8]    1.263162
(414820.8, 449389.2]    1.260407
(449389.2, 483957.6]    1.256480
(483957.6, 518526.0]    1.266944
(518526.0, 553094.4]    1.272477
(553094.4, 587662.8]    1.264204
(587662.8, 622231.2]    1.257369
(622231.2, 656799.6]    1.253732
(656799.6, 691368.0]    1.243195
Name: _row_missing, dtype: float64

**Observation:** `screen_residual` mean stays within a narrow 1.317-1.352h
band across 20 equal-sized `id` bins; `row_missing_count` mean stays within
1.243-1.272 across the same bins. Both flat, no trend -- extending Build
1's `id` finding (raw feature means, missingness, target rate all flat) to
these derived quantities. No evidence of production batches or generator
drift across the `id` range in train.

## 10. Cross-feature constraints (Phase A10)

Testing a small number of semantically-justified constraint hypotheses,
not a brute-force search over arbitrary formulas.

In [22]:
def constraint_check(name, tr_df, te_df, req_cols, violation_fn, interpretation, target_diff=None):
    tr_sub = tr_df.dropna(subset=req_cols)
    te_sub = te_df.dropna(subset=req_cols)
    tr_viol = violation_fn(tr_sub)
    te_viol = violation_fn(te_sub)
    return {
        "constraint": name,
        "train_support": len(tr_sub),
        "train_violation_rate": round(float(tr_viol), 6),
        "test_support": len(te_sub),
        "test_violation_rate": round(float(te_viol), 6),
        "target_difference": target_diff,
        "interpretation": interpretation,
    }


constraints = []

# 1. component_sum <= daily_screen_time_hours (revisits Section 4/Build 1)
req1 = ["daily_screen_time_hours"] + SCREEN_TIME_COMPONENT_COLS
tr1 = train.dropna(subset=req1)
target_diff_1 = (
    with_residual(train[train[TARGET_COLUMN] == 1])["screen_residual"].mean()
    - with_residual(train[train[TARGET_COLUMN] == 0])["screen_residual"].mean()
)
constraints.append(constraint_check(
    "daily_screen_time_hours >= social_media_hours + gaming_hours + work_study_hours",
    train, test, req1,
    lambda d: (d["daily_screen_time_hours"] < d[SCREEN_TIME_COMPONENT_COLS].sum(axis=1) - 1e-6).mean(),
    "One-sided compositional constraint, exact in both splits (0 violations). "
    "screen_residual (the slack) is the accepted Build 4 feature.",
    round(float(target_diff_1), 4),
))

# 2. weekend_screen_time vs daily_screen_time_hours
req2 = ["weekend_screen_time", "daily_screen_time_hours"]
constraints.append(constraint_check(
    "weekend_screen_time >= daily_screen_time_hours",
    train, test, req2,
    lambda d: (d["weekend_screen_time"] < d["daily_screen_time_hours"]).mean(),
    "Not a hard constraint: holds in ~85.5% of rows in both splits, consistent "
    "with an independently generated, positively correlated variable "
    "(mean ratio ~1.33), not a deterministic function of daily screen time.",
))

# 3. app_opens_per_day vs notifications_per_day
req3 = ["app_opens_per_day", "notifications_per_day"]
constraints.append(constraint_check(
    "app_opens_per_day <= notifications_per_day",
    train, test, req3,
    lambda d: (d["app_opens_per_day"] > d["notifications_per_day"]).mean(),
    "Not a hard constraint: app_opens exceeds notifications in ~30% of rows "
    "in both splits (median ratio ~0.71) -- correlated but independently "
    "generated, not one bounding the other.",
))

# 4. sleep_hours + daily_screen_time_hours <= 24h (real-world budget) -- clipping check
req4 = ["sleep_hours", "daily_screen_time_hours"]
tr4 = train.dropna(subset=req4).copy()
tr4["total"] = tr4["sleep_hours"] + tr4["daily_screen_time_hours"]
at_cap = tr4["total"] >= 19.995
near_cap_not_at = (tr4["total"] >= 19.5) & (~at_cap)
target_diff_4 = (
    train.loc[tr4.index[at_cap], TARGET_COLUMN].mean()
    - train.loc[tr4.index[near_cap_not_at], TARGET_COLUMN].mean()
)
constraints.append(constraint_check(
    "sleep_hours + daily_screen_time_hours <= 24 (real-world hour budget)",
    train, test, req4,
    lambda d: (d["sleep_hours"] + d["daily_screen_time_hours"] > 24).mean(),
    "Never violated in either split, but the sum empirically caps at ~20.0h, "
    "not 24h, with a visible frequency spike exactly at 20.00 -- a clipping "
    "artifact, not a real 24h-day constraint (see cell below).",
    round(float(target_diff_4), 4),
))

constraints_df = pd.DataFrame(constraints)
constraints_df

,constraint,train_support,train_violation_rate,test_support,test_violation_rate,target_difference,interpretation
0,daily_screen_time_hours >= social_media_hours ...,421427,0.000000,182287,0.000000,0.9943,"One-sided compositional constraint, exact in b..."
1,weekend_screen_time >= daily_screen_time_hours,517906,0.145395,225410,0.144758,NaN,Not a hard constraint: holds in ~85.5% of rows...
2,app_opens_per_day <= notifications_per_day,568082,0.299399,245890,0.300756,NaN,Not a hard constraint: app_opens exceeds notif...
3,sleep_hours + daily_screen_time_hours <= 24 (r...,559350,0.000000,244527,0.000000,0.0002,"Never violated in either split, but the sum em..."


In [23]:
# Quantify the clipping spike found for constraint #4
print("frac train complete rows within 0.5h of the 20h cap:", round((tr4["total"] >= 19.5).mean(), 4))
spike_at_20 = tr4["total"].round(2).value_counts().sort_index(ascending=False).head(10)
print(spike_at_20)
print(
    "\ntarget rate at cap (>=19.995h):", round(train.loc[tr4.index[at_cap], TARGET_COLUMN].mean(), 4),
    "vs just below cap (19.5-19.99h):", round(train.loc[tr4.index[near_cap_not_at], TARGET_COLUMN].mean(), 4),
)
print("correlation(sleep_hours, daily_screen_time_hours):", round(tr4["sleep_hours"].corr(tr4["daily_screen_time_hours"]), 4))
constraints_df.to_csv(OUTPUTS_DIR / "generator_constraints.csv", index=False)

frac train complete rows within 0.5h of the 20h cap: 0.044
total
20.01      232
20.00    14132
19.99      371
19.98      169
19.97      206
19.96      208
19.95      154
19.94      187
19.93      168
19.92      172
Name: count, dtype: int64

target rate at cap (>=19.995h): 0.9997 vs just below cap (19.5-19.99h): 0.9995
correlation(sleep_hours, daily_screen_time_hours): 0.0282


**Observation:** the sleep+screen-time sum never exceeds 24h in either
split, but empirically caps at 20.00-20.01h with a sharp spike exactly at
20.00 (14,132 of 559,350 = 2.53% of train complete rows, vs a few hundred
at each neighboring 0.01 step). `sleep_hours` and `daily_screen_time_hours`
are nearly uncorrelated (r=0.03), so this reads as independent draws with a
post-hoc joint clip -- a **real, measurable generator fingerprint**. But
target rate at the cap (0.9997) is statistically indistinguishable from
rows just below it (0.9995): the effect is fully explained by
`daily_screen_time_hours` already being near its own maximum at the cap,
not an independent signal from the clip. The other two candidate
constraints (`weekend_screen_time`, `app_opens_per_day`) are not hard
constraints at all -- correlated but independently generated variables.

Artifact: `outputs/generator_constraints.csv`.

## 11. Possible source-data fingerprints (Phase A11)

Synthesizing Sections 5-10 plus Build 1's duplicate audit into a single
assessment of whether the dataset looks like it was generated from a
smaller latent/source dataset (repeated templates, sharp exact-value
target rates, duplicated profiles) versus a large or fully continuous
parametric generator.

| Signal checked | Result |
|---|---|
| Exact predictor-vector duplicates (Build 1, full predictor set) | None within train or test |
| Screen-time 4-column combination, native (0.01) precision | 99.99% unique (Section 6) |
| Screen-time 4-column combination, 0.1h rounding | Coincidental small clusters, no target consistency (Section 7) |
| Missingness pattern space | 2,925/~4,096 realized, consistent with independent per-column draws (Section 8) |
| Exact-value target rates (`screen_residual`, rounded `daily_screen_time_hours`) | Smooth monotonic gradients, not abrupt jumps at specific values (Section 5) |

**Assessment:** *does not support* reconstructing or exploiting a smaller
latent source dataset. Every check that could reveal template reuse
(native-precision uniqueness, coarser-rounding target consistency,
missingness pattern diversity) instead shows behavior consistent with a
large or fully continuous parametric generator sampling each feature
(largely) independently, subject to the documented compositional
(`screen_residual`) and clipping (sleep+screen cap) relationships. This is
a *weak-evidence* verdict in the terminology of Section 12, not a
"no evidence" verdict, because the compositional and clipping constraints
are themselves real, non-trivial generator structure -- they are just not
evidence of a *small, reusable* source dataset.

## 12. Rules and risk assessment

Reviewed the competition rules surface available locally
(`docs/COMPETITION_NOTES.md`) before considering any unusual technique.
No local record of Playground S6E8's specific rules text beyond what has
already been captured; the standard Kaggle Playground Series terms
(no external private data, no leaked/hidden labels, no sharing of private
outputs between competitors) are assumed to apply as with prior builds.
**Not independently re-verified against the live Kaggle rules page in this
session** -- if a Bucket 3/4 technique were ever proposed, that page should
be checked first. No such technique is proposed here (see table below), so
this was not blocking for Build 5.

| Technique considered | Bucket | Transductive? | Target-derived? | Leakage risk | Rules status | Decision |
|---|---|---|---|---|---|---|
| `screen_residual` (existing, Build 4) | 1 (standard engineered feature) | No | No | None | Clearly allowed | Already accepted; re-characterized, not changed |
| Value-frequency encoding (`age`, `notifications_per_day`, `app_opens_per_day`) | 1/2 (train-only vs train+test) | Would be, if built train+test | No | Low | Clearly allowed if train-only; likely allowed but worth caution if train+test | Not pursued -- no target correlation found (Section 5) |
| Missingness-pattern encoding (12-bit mask or pattern count) | 1 (standard engineered feature) | No | No | None | Clearly allowed | Not pursued -- flat target rate (Section 8) |
| `sleep_hours + daily_screen_time_hours` at-cap flag | 1 (standard engineered feature) | No | No | None | Clearly allowed | Not pursued -- redundant with `daily_screen_time_hours` (Section 10) |
| 0.1h-rounded profile -> target lookup/propagation | 3 (lookup-like / source reconstruction) | No (train-only) but target-derived | Yes | High (overfitting) | Not appropriate given the evidence | Rejected -- group target rates span the full 0-1 range even at 5+ row support (Section 7) |
| Exact-value target encoding on `screen_residual` or rounded `daily_screen_time_hours` | 3 (lookup-like) if not OOF; 1 if properly OOF | No | Yes | Moderate if not fold-safe | Would require fold-safe/OOF construction and strong justification | Not pursued -- Section 5 shows the relationship is smooth and already well-captured by the raw continuous feature in a tree model; no expected gain over what XGBoost/CatBoost already extract |

**No Bucket 4 (clearly inappropriate) technique was found, proposed, or
considered.** No external private data, leaked labels, hidden test labels,
other competitors' outputs, or unauthorized source reconstruction were
used anywhere in this notebook.

## 13. Candidate exploitation hypotheses

Ranked candidate list from Phase A. None reached the bar for a Phase B
formal experiment (see Section 14).

| Hypothesis ID | Observed generator behavior | Proposed feature/technique | Why it might help | Leakage risk | Transductive? | Rules concern | Expected model interaction | Priority |
|---|---|---|---|---|---|---|---|---|
| G001 | `screen_residual` grid/continuity (Section 4) | (already implemented as `screen_residual`, Build 4) | Explains, does not extend, the existing accepted feature | None | No | None | Already captured by E006 | N/A -- not a new candidate |
| G002 | Missingness pattern diversity (Section 8) | Missing-pattern mask/count feature | Target rate is flat across patterns -- no expected gain | None | No | None | None expected | Rejected |
| G003 | Near-duplicate 0.1h-rounded profiles (Section 7) | Profile repeat-count or target-lookup feature | Target rate within repeat groups spans 0-1 -- would fit noise, not signal | High (overfitting) | No | Not appropriate | Likely to hurt CV via overfit | Rejected |
| G004 | Value frequency stability (Section 5) | Train (or train+test) frequency-count feature on `age`/`notifications_per_day`/`app_opens_per_day` | No frequency-target correlation found | Low/none (train-only); would need documentation if train+test | Only if train+test | Clearly allowed if train-only | None expected | Rejected -- no evidence of gain |
| G005 | Sleep+screen-time clipping spike (Section 10) | At-cap binary flag | Real generator fingerprint but fully redundant with `daily_screen_time_hours` | None | No | None | No gain expected over raw feature | Rejected |

**Every candidate was rejected or judged non-actionable.** This is
recorded as a Build 5 finding in its own right (`outputs/synthetic_generator_findings.csv`),
not as an incomplete investigation.

## 14. Optional controlled experiments

Phase A did not surface a candidate meeting the bar stated in the Build 5
brief ("a strong, explainable, rules-compliant signal"). Every candidate
in Section 13 was rejected on Phase A evidence alone (no target
correlation, redundant with an existing feature, or high overfitting
risk), so **no formal Phase B experiment is run in this build**. This is
one of the two explicitly valid Build 5 outcomes: *"synthetic structure
exists, but no safe/credible feature worth using was identified."*

The frozen control (E006, CV mean 0.96445, public LB 0.96608) is carried
forward unchanged into Build 6.

## 15. Build 5 conclusions

**Strongest generator findings** (full detail: `outputs/synthetic_generator_findings.csv`):

1. **F01 -- screen-time composition (real, understood, already exploited).**
   `daily_screen_time_hours >= component_sum` holds exactly in both splits;
   the slack (`screen_residual`) is a continuous, right-skewed,
   train/test-consistent quantity, strongly and smoothly related to the
   target. This is the mechanism behind the accepted Build 4 feature, not
   a new opportunity.
2. **F02 -- sleep/screen-time clipping near 20h (real generator fingerprint,
   not actionable).** A genuine clipping artifact (spike at exactly 20.00h),
   but fully confounded with `daily_screen_time_hours`, which the models
   already use directly.
3. **F03 -- rounded `daily_screen_time_hours` staircase (ordinary
   predictive relationship, not an artifact).** The dataset's dominant
   signal, already captured optimally by tree-based splits.
4. **F05-F08 -- no repeated-profile, near-duplicate, or missingness-pattern
   structure found** that would support any lookup-like or template-reuse
   technique. Everywhere checked, the evidence argues against a small
   reusable source dataset.

**Quantization:** all continuous predictors sit on a clean 0.01 grid
(negligible float noise, <5 rows per split); count-like features
(`age`, `notifications_per_day`, `app_opens_per_day`) are exact integer
grids. No hidden finer- or coarser-grid structure found. Train and test
use identical grids and ranges.

**Repeated-pattern findings:** no exact or near-duplicate structure
supports a lookup or template-reuse feature. The 4-column screen-time
combination is 99.99% unique at native precision, and 0.1h-rounded groups
show target rates spanning the full 0-1 range even at 5+ row support.

**Missingness-generator findings:** 2,925 of a ~4,096-pattern space
realized, consistent with independent per-column missingness; target rate
flat across the top 15 patterns and across `row_missing_count` for all
well-supported counts. No encoding justified.

**ID/batch findings:** no drift in `screen_residual` or
`row_missing_count` across 20 `id` bins -- extends Build 1's `id` finding,
no batching evidence.

**Source-data fingerprint assessment: Weak evidence.** Real generator
structure exists (screen-time composition, sleep/screen clipping) but
every check for template reuse or a small latent source dataset came back
negative.

**Rules/transductive decisions:** no transductive or target-derived
feature was implemented. The one train+test-frequency idea considered
(G004) was rejected on evidence before any rules question became live.

**Formal Build 5 experiments:** none run -- no candidate cleared the Phase
A bar. E006 (XGBoost + `screen_residual`) remains the best validated model
and best public LB score.

**Accepted generator-inspired features (this build):** none new.
`screen_residual` (Build 4) remains the sole accepted generator-derived
feature, now with its generation mechanism characterized.

**Rejected generator-inspired ideas (this build):** missingness-pattern
encoding (G002), near-duplicate/repeat-count or target-lookup features
(G003), frequency encoding on `age`/`notifications_per_day`/`app_opens_per_day`
(G004), sleep+screen-time at-cap flag (G005).

**Next build:** Build 6 - Controlled Hyperparameter Tuning, starting from
the frozen E006 control and the known CatBoost iteration-budget headroom
(Build 3 finding, `best_iteration=799` every fold at the 800-iteration cap).